In [1]:
import numpy as np
import sqlite3
import pandas as pd

In [4]:
conn = sqlite3.connect("../../financial-data-pipeline/data/market.db")
spy = pd.read_sql_query(
        "SELECT date, adj_close FROM prices WHERE ticker = 'SPY' ORDER BY date", conn,
        parse_dates=["date"], index_col="date"
    )
conn.close()

In [5]:
spy.head()

,adj_close
date,
2014-01-02,147.831345
2014-01-03,147.807159
2014-01-06,147.378738
2014-01-07,148.283936
2014-01-08,148.316269


In [6]:
spy.index.dtype

dtype('<M8[us]')

In [7]:
len(spy)

3202

In [8]:
print(spy.index.min(), spy.index.max())

2014-01-02 00:00:00 2026-09-25 00:00:00


In [9]:
spy['daily_return']=spy["adj_close"].pct_change()

In [20]:
spy.head()

,adj_close,daily_return,ma50,signal,position,strat_ret
date,,,,,,
2014-01-02,147.831345,NaN,NaN,0,NaN,NaN
2014-01-03,147.807159,-0.000164,NaN,0,0.0,-0.0
2014-01-06,147.378738,-0.002899,NaN,0,0.0,-0.0
2014-01-07,148.283936,0.006142,NaN,0,0.0,0.0
2014-01-08,148.316269,0.000218,NaN,0,0.0,0.0


In [11]:
spy['ma50'] = spy['adj_close'].rolling(window=50).mean()

In [16]:
spy['signal'] = np.where(spy['adj_close'] > spy['ma50'], 1, 0)

In [17]:
spy['signal'].value_counts()

signal
1    2307
0     895
Name: count, dtype: int64

In [18]:
spy.iloc[45:55]

,adj_close,daily_return,ma50,signal
date,,,,
2014-03-10,152.066193,-0.000532,NaN,0
2014-03-11,151.314529,-0.004943,NaN,0
2014-03-12,151.355042,0.000268,NaN,0
2014-03-13,149.657883,-0.011213,NaN,0
2014-03-14,149.237656,-0.002808,147.851253,1
2014-03-17,150.587234,0.009043,147.906371,1
2014-03-18,151.662094,0.007138,147.983470,1
2014-03-19,150.853943,-0.005329,148.052974,1
2014-03-20,151.734863,0.005840,148.121992,1


In [19]:
spy['position']=spy['signal'].shift(1)
spy['strat_ret']=spy['position']*spy['daily_return']

In [21]:
spy.iloc[1000:1010]

,adj_close,daily_return,ma50,signal,position,strat_ret
date,,,,,,
2017-12-20,233.846008,-0.000524,226.676284,1,1.0,-0.000524
2017-12-21,234.327698,0.002060,226.918981,1,1.0,0.002060
2017-12-22,234.266418,-0.000262,227.167075,1,1.0,-0.000262
2017-12-26,233.986145,-0.001196,227.404161,1,1.0,-0.001196
2017-12-27,234.099976,0.000486,227.637599,1,1.0,0.000486
2017-12-28,234.581680,0.002058,227.877536,1,1.0,0.002058
2017-12-29,233.697174,-0.003771,228.095425,1,1.0,-0.003771
2018-01-02,235.369843,0.007157,228.345548,1,1.0,0.007157
2018-01-03,236.858627,0.006325,228.602445,1,1.0,0.006325


In [22]:
spy[spy['signal'].diff() != 0]

,adj_close,daily_return,ma50,signal,position,strat_ret
date,,,,,,
2014-01-02,147.831345,NaN,NaN,0,NaN,NaN
2014-03-14,149.237656,-0.002808,147.851253,1,0.0,-0.000000
2014-04-10,148.678680,-0.021006,149.256538,0,1.0,-0.021006
2014-04-16,151.089554,0.010478,149.758465,1,0.0,0.000000
2014-07-31,157.490311,-0.019748,159.030896,0,1.0,-0.019748
...,...,...,...,...,...,...
2026-07-31,745.179626,0.007200,742.378932,1,0.0,0.000000
2026-09-10,755.952820,-0.005994,756.367593,0,1.0,-0.005994
2026-09-11,762.396790,0.008524,756.737274,1,0.0,0.000000
